## Evaluation pipeline for the microlane experiment

In [7]:
# First consider all the variables
# The input image gets resized to a particular level
# Then create a pipeline to feed data into the model
# AFter this process is completed, then process the data
# Then after the processing is done find a way to take output from the model
# Then, convert the output to relevant format, and store it for future use
# Apply relevant computations

In [8]:
# Imports of the Core Packages
import json, sys, time, pytz
import os, yaml,random 
import numpy as np
from datetime import datetime
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
# Import custom libraries located at different folder location + configs
from microlane.utils.metrics import *
from microlane.datasets.tusimple import TuSimple
from microlane.models.lanenet2.model import LaneNet2
from microlane.schema.output import ModelPrediction
from microlane.schema.sample import Sample
from microlane.utils.load_image import load_image_from_sample
from microlane.utils.experiment import ExperimentEvaluate

ModuleNotFoundError: No module named 'microlane'

In [ ]:
# First Load the Configuation file
with open("configs/config.yaml", "r") as file:
    config = yaml.safe_load(file)

### Pre Processing Part

In [ ]:
# First initialise the dataset
# Then load the dataset
dataset = TuSimple(
    
        folder_path=config['data']['datasets']['tusimple']['path'],
        
        annotation_file_path=config['data']['datasets']['tusimple']['annotation_file']
    )

data = dataset.load(number=1000)

In [ ]:
# So, basically now we will import the model
# model = LaneNet2() type and what we will do is, run 
# Run model.inference(formatted_dataset)
# Ensure that Docker Engine Is Running in background

model = LaneNet2(
    
    container_folder=config['models']['lanenet2']['container_folder'],
    
    image_name=config['models']['lanenet2']['image_name']
    
)

In [ ]:
experiment = ExperimentEvaluate(
    
    experiment_name="pipeline testing with no augmentation"
    
)

### Models and Datasets Loaded, Now Processing Part

In [ ]:
# Print some basic information of our data

print(f"Total items: {len(data)}\n")

random_image_index = random.randint(0, len(data)-1)

item = data[21  ]
print(f"Index        : {random_image_index}")
print(f"Image Path   : {item.image_path}")
print(f"h_samples    : {item.h_samples}")
print(f"lanes        : {item.lanes}")

### Adding Image Augmentation

In [ ]:
blur_range = tuple(config['data']['augmentation']['blur'])
rotation_range = tuple(config['data']['augmentation']['blur'])
zoom_range = tuple(config['data']['augmentation']['zoom'])
brightness_range =tuple(config['data']['augmentation']['brightness'])

In [ ]:
# item.blur       = round(random.uniform(blur_range[0],       blur_range[1] - 0.01), 2)
# item.rotation   = round(random.uniform(rotation_range[0],   rotation_range[1] - 0.01), 2)
# item.zoom       = round(random.uniform(zoom_range[0],       zoom_range[1] - 0.01), 2)
# item.brightness = round(random.uniform(brightness_range[0], brightness_range[1] - 0.01), 2)

print(f"Blur:       {item.blur:.2f}")
print(f"Rotation:   {item.rotation:.2f}")
print(f"Zoom:       {item.zoom:.2f}")
print(f"Brightness: {item.brightness:.2f}")

In [ ]:

loaded_image = load_image_from_sample(item)

In [ ]:
if loaded_image.image is not None:
    plt.imshow(loaded_image.image)
    plt.axis("off")
    plt.title("Initial Image")
    plt.show()

In [ ]:
# We are basically sending a loaded sample with actual image tensor in the memory

response = model.predict(loaded_image)

In [ ]:
prediction = response.json()

ModelOutput = ModelPrediction (
    
    sample=Sample(
        
            image=prediction['sample']['image'],
            image_path=prediction['sample']['image_path'],
            h_samples=prediction['sample']['h_samples'],
            lanes=prediction['sample']['lanes'],
            blur=prediction['sample']['blur'],
            brightness=prediction['sample']['brightness'],
            zoom=prediction['sample']['zoom'],
            rotation=prediction['sample']['rotation']

        ),
    
    lanes=prediction['lanes'],
        
    run_time=prediction['run_time']
    
)

In [ ]:

experiment.store_prediction(ModelOutput)

In [ ]:
experiment.visualize_prediction(ModelOutput, show=True)